In [1]:
import torch
import gc
import re
import json
import pandas as pd
from tqdm import tqdm
from transformers import AutoModelForCausalLM, BitsAndBytesConfig, AutoTokenizer
from peft import PeftModel

# --- MEMORY CLEANER UTILITY ---
def clean_memory():
    """Aggressively clears VRAM."""
    if 'model' in globals(): del globals()['model']
    if 'base_model' in globals(): del globals()['base_model']
    if 'tokenizer' in globals(): del globals()['tokenizer']
    gc.collect()
    torch.cuda.empty_cache()
    print("✨ Memory Cleared")

# --- ANSWER EXTRACTION LOGIC ---
def extract_answer(text):
    """
    Robust extraction looking for 'Answer: X' or 'Final Answer: X'.
    Returns the *last* found match to avoid catching examples in the prompt.
    """
    # Look for 'answer' followed by optional colon/hyphen and a letter a-e
    matches = re.findall(r'(?:final|answer)\s*[:\-]?\s*([a-e])', text.lower())
    if matches:
        return matches[-1] # Return the last one found
    return "None"

# --- MAIN EVALUATION FUNCTION ---
def evaluate_model(model_path, test_df, num_samples=50):
    """
    Loads a model, evaluates it on N samples, calculates accuracy,
    saves results, and then UNLOADS the model to free memory.
    """
    print(f"\n🚀 STARTING EVALUATION FOR: {model_path}")

    # 1. Load Tokenizer & Config
    # Using 4-bit to ensure inference is fast and memory-safe
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
    )

    model_id = "mistralai/Mistral-7B-Instruct-v0.2"
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    tokenizer.pad_token = tokenizer.eos_token

    # 2. Load Model
    print("   Loading Base Model...")
    base_model = AutoModelForCausalLM.from_pretrained(
        model_id,
        quantization_config=bnb_config,
        device_map={"": "cuda:0"}
    )

    print(f"   Loading Adapters from {model_path}...")
    try:
        model = PeftModel.from_pretrained(base_model, model_path)
    except Exception as e:
        print(f"❌ Error loading adapter: {e}")
        return None

    model.eval()

    # 3. Prepare Data
    # Sample N random rows (fixed random_state for consistency across models)
    subset = test_df.sample(n=num_samples, random_state=42)

    results = []
    correct_count = 0

    print(f"   Generating responses for {num_samples} examples...")

    # 4. Inference Loop
    for _, row in tqdm(subset.iterrows(), total=num_samples):
        raw_problem = row['Problem']
        ground_truth = row['correct'].strip().lower()

        # Exact prompt format used in training
        prompt = (
            f"Question: {raw_problem}\n"
            f"Options: {row['options']}\n"
            "Answer this question by providing a Rationale followed by the Final Answer.\n\n"
            "### Response:\n"
        )

        inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=128,
                pad_token_id=tokenizer.eos_token_id,
                do_sample=False # Greedy decoding for consistent evaluation
            )

        # Decode and strip prompt
        full_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
        response_text = full_text.replace(prompt, "").strip()

        # Extract and Check
        extracted = extract_answer(response_text)
        is_correct = (extracted == ground_truth)

        if is_correct:
            correct_count += 1

        results.append({
            "model": model_path,
            "problem_snippet": raw_problem[:50],
            "ground_truth": ground_truth,
            "extracted": extracted,
            "is_correct": is_correct,
            "full_response": response_text
        })

    # 5. Metrics
    accuracy = (correct_count / num_samples) * 100
    print(f"\n📊 {model_path} Accuracy: {accuracy:.2f}%")

    # 6. Cleanup (CRITICAL)
    del model
    del base_model
    del tokenizer
    gc.collect()
    torch.cuda.empty_cache()

    return pd.DataFrame(results), accuracy

In [2]:
!pip install trl
!pip install -U bitsandbytes

In [3]:
from google.colab import drive

drive.mount('/content/drive')
# Load Data
val_list = json.load(open('/content/drive/MyDrive/CIS 602 Project 2/data/test.json'))
val_df = pd.DataFrame(val_list)


# --- EVALUATE SFT MODEL ---
sft_model_path = "/content/drive/MyDrive/CIS 602 Project 2/RLHF/sft_warmup_model"
sft_df, sft_acc = evaluate_model(sft_model_path, val_df, num_samples=50)

# --- EVALUATE RL MODEL ---
rl_model_path = "/content/drive/MyDrive/CIS 602 Project 2/RLHF/model3"
rl_df, rl_acc = evaluate_model(rl_model_path, val_df, num_samples=50)

# --- DISPLAY COMPARISON ---
print("\n" + "="*40)
print("     FINAL HEAD-TO-HEAD RESULTS")
print("="*40)
print(f"SFT Warmup Accuracy:  {sft_acc:.2f}%")
print(f"RL (GRPO) Accuracy:   {rl_acc:.2f}%")

delta = rl_acc - sft_acc
print(f"Improvement:          {delta:+.2f}%")

# Create a merged view for qualitative analysis
if sft_df is not None and rl_df is not None:
    comparison_view = sft_df[['problem_snippet', 'ground_truth']].copy()
    comparison_view['SFT_Answer'] = sft_df['extracted']
    comparison_view['SFT_Correct'] = sft_df['is_correct']
    comparison_view['RL_Answer'] = rl_df['extracted']
    comparison_view['RL_Correct'] = rl_df['is_correct']

    # Show cases where RL fixed a mistake
    improved = comparison_view[
        (comparison_view['SFT_Correct'] == False) &
        (comparison_view['RL_Correct'] == True)
    ]

    if not improved.empty:
        print(f"\nFound {len(improved)} cases where RL fixed the error:")
        display(improved.head())
    else:
        print("\nNo cases found where RL fixed an SFT error in this sample batch.")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

🚀 STARTING EVALUATION FOR: /content/drive/MyDrive/CIS 602 Project 2/RLHF/sft_warmup_model


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


   Loading Base Model...


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

...


📊 /content/drive/MyDrive/CIS 602 Project 2/RLHF/sft_warmup_model Accuracy: 14.00%

🚀 STARTING EVALUATION FOR: /content/drive/MyDrive/CIS 602 Project 2/RLHF/model3
   Loading Base Model...


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

   Loading Adapters from /content/drive/MyDrive/CIS 602 Project 2/RLHF/model3...
   Generating responses for 50 examples...


100%|██████████| 50/50 [09:05<00:00, 10.91s/it]



📊 /content/drive/MyDrive/CIS 602 Project 2/RLHF/model3 Accuracy: 14.00%

     FINAL HEAD-TO-HEAD RESULTS
SFT Warmup Accuracy:  14.00%
RL (GRPO) Accuracy:   14.00%
Improvement:          +0.00%

No cases found where RL fixed an SFT error in this sample batch.
